### ファイルインポート

In [51]:
import pandas as pd
import numpy as np


In [57]:
# --- ファイル読み込み ---

df_agg = pd.read_parquet("liq_engine_agg.parquet", engine="pyarrow")
df_feats = pd.read_parquet("liq_engine_feats.parquet", engine="pyarrow")


In [58]:
forward_weeks = 8

In [79]:
# データセット作成
df = df_feats[[
    "Liq_eff","^GSPC", "NDFACBM027SBOG_z52", "Cu_Au_Ratio_z252", "Net_Liquidity_z252","DX-Y.NYB"]].dropna()
df['forward_return'] = df['^GSPC'].shift(-forward_weeks)  / df['^GSPC']- 1
df['Liq_Regime'] = pd.qcut(
    df['DX-Y.NYB'], 5,
    labels=['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)'])

plot_df = df.dropna(subset=['forward_return', 'Liq_Regime'])

df_feats.columns


Index(['^GSPC', 'TLT', 'BAMLH0A0HYM2', 'RRPONTSYD', 'DTB3', 'DFF', 'DX-Y.NYB',
       'HG=F', 'GC=F', 'DFII10', 'THREEFYTP10', 'UUP', 'WALCL', 'WDTGAL',
       'WCURCIR', 'NFCI', 'TOTBKCR', 'M2SL', 'INDPRO', 'NDFACBM027SBOG',
       'BUSLOANS', 'RRP_filled', 'Net_Liquidity', 'Net_Liquidity_z252',
       'NFCI_z252', 'Liq_eff', 'Liq_eff_ma5', 'Liq_eff_diff20',
       'Marshallian_K', 'Credit_Growth_z252', 'Cu_Au_Ratio_z252', 'UUP_z52',
       'NDFACBM027SBOG_z52', 'Real_Rate_Gravity', 'Term_Premium',
       'HY_Spread_Momentum', 'DTB3_DFF_Spread', 'DXY_roc65'],
      dtype='str')

In [80]:
import plotly.express as px
import plotly.graph_objects as go

fig = px.box(
    plot_df, 
    x='Liq_Regime', 
    y='forward_return',
    color='Liq_Regime',
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by Liq_eff Regime",
    labels={'forward_return': f'{forward_weeks}-Week Forward Return', 'Liq_Regime': 'Liquidity Regime'},
    category_orders={"Liq_Regime": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show(renderer="browser",config=dict(displayModeBar=False))
